In [ ]:
"""
Figure 4.3 - Bachelor Thesis on the Elephant Random Walk.

Diffusive regime (p < 3/4). Histogram of the rescaled walk
        Z_n := S_n / sqrt(n / (3 - 4p)),
over 8,000 paths at n = 7,000, q = 1/2, for p in {0.35, 0.65},
with the N(0, 1) density overlaid.

At q = 1/2 the mean-centering term vanishes (beta = 0), so the rescaling
simplifies to what is written above (Theorem 4.6).
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import norm

# ── Fonts ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.labelsize":     9,
    "axes.titlesize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "figure.dpi":         150,
})

# ── Thesis palette (matching Figures 4.4 - 4.8) ──────────────────────────────
TEAL = ( 30/255, 130/255, 130/255)
RED  = (177/255,  56/255,  69/255)

PANELS = [
    (0.35, TEAL),
    (0.65, RED),
]

# ── Bar colour helpers (pale toward the tails) ────────────────────────────────
def _bar_color(rgb, x, x_max, pale_mix=0.72):
    t = min(abs(x) / x_max, 1.0) if x_max > 0 else 0.0
    return tuple(c + (1 - c) * pale_mix * t for c in rgb)

def _edge_color(rgb, x, x_max, pale_mix=0.35):
    t = min(abs(x) / x_max, 1.0) if x_max > 0 else 0.0
    return tuple(c + (1 - c) * pale_mix * t for c in rgb)

# ── ERW simulator ────────────────────────────────────────────────────────────
def simulate_erw_batch(n, p, q, n_sims, rng=None):
    """Simulate n_sims ERW paths of length n and return the final positions S_n."""
    if rng is None:
        rng = np.random.default_rng()
    xi = np.zeros((n_sims, n + 1), dtype=np.int8)
    xi[:, 1] = np.where(rng.random(n_sims) < q, 1, -1)
    for t in range(1, n):
        idx  = rng.integers(1, t + 1, size=n_sims)
        past = xi[np.arange(n_sims), idx]
        flip = np.where(rng.random(n_sims) < p, 1, -1)
        xi[:, t + 1] = past * flip
    return xi[:, 1:].sum(axis=1).astype(float)

# ── Parameters (Figure 4.3) ───────────────────────────────────────────────────
N_STEPS = 7_000
N_SIMS  = 50_000
N_BINS  = 60
Q       = 0.50   # beta = 0

rng = np.random.default_rng(314159)

# ── Simulate and apply diffusive-regime scaling ──────────────────────────────
# Z_n = S_n / sqrt(n / (3 - 4p))
finals = {}
for p_val, _ in PANELS:
    raw = simulate_erw_batch(N_STEPS, p=p_val, q=Q, n_sims=N_SIMS, rng=rng)
    scale_denom = np.sqrt(N_STEPS / (3 - 4 * p_val))
    finals[p_val] = raw / scale_denom

# ── Axis limits shared across the two panels ──────────────────────────────────
x_lim = 3.5
bin_edges = np.linspace(-x_lim, x_lim, N_BINS + 1)

all_counts = {}
for p_val, _ in PANELS:
    c, _ = np.histogram(finals[p_val], bins=bin_edges, density=True)
    all_counts[p_val] = c

x_fine    = np.linspace(-x_lim, x_lim, 600)
gauss_pdf = norm.pdf(x_fine)

y_max = max(max(c.max() for c in all_counts.values()), gauss_pdf.max()) * 1.08

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(5.2, 2.9))
fig.subplots_adjust(left=0.12, right=0.98, bottom=0.20, top=0.86, wspace=0.30)

LABEL_GREY = (0.55, 0.55, 0.55)

bin_width   = bin_edges[1] - bin_edges[0]
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

for ax, (p_val, color) in zip(axes, PANELS):
    counts = all_counts[p_val]

    for xc, h in zip(bin_centers, counts):
        ax.bar(xc, h, width=bin_width,
               color=_bar_color(color, xc, x_lim),
               edgecolor=_edge_color(color, xc, x_lim),
               linewidth=0.28, alpha=0.70, zorder=2)

    # Gaussian overlay
    ax.plot(x_fine, gauss_pdf, color=color, lw=1.0, zorder=3)

    ax.set_title(fr"$p\!=\!{p_val:.2f}$", fontsize=9, pad=5, color=LABEL_GREY)
    ax.set_xlabel(r"$Z_n$", labelpad=3)
    ax.set_xlim(-x_lim, x_lim)
    ax.set_ylim(0, y_max)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.55)
    ax.spines["bottom"].set_linewidth(0.55)
    ax.tick_params(width=0.55, length=3, direction="out")

    ax.xaxis.set_major_locator(ticker.FixedLocator([-3, 0, 3]))
    ax.xaxis.set_major_formatter(
        ticker.FuncFormatter(lambda x, _:
            r"$-3$" if x == -3 else (r"$3$" if x == 3 else r"$0$"))
    )
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))

# Shared y-axis: hide y-tick labels on the right panel
axes[1].set_yticks(axes[0].get_yticks())
axes[1].set_ylim(0, y_max)
axes[1].tick_params(axis="y", labelleft=False)

axes[0].set_ylabel("Density", labelpad=4, fontsize=8)

fig.savefig("figure_4_3.pdf", bbox_inches="tight", pad_inches=0.02)
print("Saved figure_4_3.pdf")